## In-context learning

here we show an example of how in-context learning can change the output of a large language model.

In [1]:
# Import the os package
import os

# Import the openai package
from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI"),
)


In [2]:
# Define the system message
system_msg = 'You are a helpful assistant who understands Python programming.'

# Define the user message
user_msg = 'generate a python function to compute a multiple linear regression solution using linear algebra.' 
content_plain = []


def run_gpt(input, nruns=20):
    print(f"Input:", input)
    content = []
    for i in range(nruns):
        # Create a dataset using GPT
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": input
                }
            ],
            model="gpt-5.2",
        )
        content.append(chat_completion.to_dict()['choices'][0]['message']['content'].split("```"))
    return content

content_plain = run_gpt(user_msg, 20)

Input: generate a python function to compute a multiple linear regression solution using linear algebra.


In [3]:
content_context = []

nruns = 20

content1 = "why are type hints important when creating a python function?"
content2 = "generate a python function to compute a multiple linear regression solution using linear algebra."

for i in range(nruns):
    print(f"Run {i+1}/{nruns}")
    # Create a dataset using GPT

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": content1
            },
            {
                "role": "user",
                "content": "generate a python function to compute a multiple linear regression solution using linear algebra.",
            }
        ],
        model="gpt-5.2",
    )
    content = chat_completion.to_dict()['choices'][0]['message']['content'].split("```")
    content_context.append(content)


Run 1/20
Run 2/20
Run 3/20
Run 4/20
Run 5/20
Run 6/20
Run 7/20
Run 8/20
Run 9/20
Run 10/20
Run 11/20
Run 12/20
Run 13/20
Run 14/20
Run 15/20
Run 16/20
Run 17/20
Run 18/20
Run 19/20
Run 20/20


In [4]:
for i, resp in enumerate(content_plain):
    print(f"Run {i+1}: ", [x for x in resp[1].split('\n') if 'def' in x][0])



Run 1:  def multiple_linear_regression(X, y, *, add_intercept=True, method="solve"):
Run 2:  def multiple_linear_regression(X, y, fit_intercept=True, method="auto", rcond=None):
Run 3:  def multiple_linear_regression(X, y, add_intercept=True, method="solve"):
Run 4:  def multiple_linear_regression(X, y, fit_intercept=True, rcond=None):
Run 5:  def multiple_linear_regression(X, y, fit_intercept=True, method="solve"):
Run 6:  def multiple_linear_regression(X, y, add_intercept=True, method="svd", rcond=None):
Run 7:  def multiple_linear_regression(X, y, add_intercept=True, method="solve"):
Run 8:  def multiple_linear_regression(X, y, add_intercept=True, method="auto"):
Run 9:  def multiple_linear_regression(X, y, add_intercept=True, method="qr", rcond=None):
Run 10:  def multiple_linear_regression(X, y, add_intercept=True, method="lstsq"):
Run 11:  def multiple_linear_regression(X, y, add_intercept=True, method="solve"):
Run 12:  def multiple_linear_regression(X, y, add_intercept=True, me

In [5]:
for i, resp in enumerate(content_context):
    print(f"Run {i+1}: ", [x for x in resp[1].split('\n') if 'def' in x])


Run 1:  ['def linear_regression_ols(', '    add_intercept : bool, default=True']
Run 2:  ['def multiple_linear_regression(']
Run 3:  ['def linear_regression_ols(']
Run 4:  ['def multiple_linear_regression(']
Run 5:  ['def linear_regression_ols(']
Run 6:  ['def linear_regression_ols(', '    add_intercept : bool, default=True']
Run 7:  ['def linear_regression_ols(']
Run 8:  ['def multiple_linear_regression(']
Run 9:  ['def multiple_linear_regression(']
Run 10:  ['def multiple_linear_regression(', '    add_intercept : bool, default=True']
Run 11:  ['def multiple_linear_regression(']
Run 12:  ['def multiple_linear_regression(']
Run 13:  ['def linear_regression_ols(', '    add_intercept : bool, default True']
Run 14:  ['def multiple_linear_regression(']
Run 15:  ['def multiple_linear_regression(', '    add_intercept : bool, default=True']
Run 16:  ['def multiple_linear_regression(']
Run 17:  ['def multiple_linear_regression(', '    fit_intercept : bool, default=True']
Run 18:  ['def multipl

In [6]:
def extract_full_function_signature(code_text):
    """Extract the full function signature from code, handling multi-line signatures."""
    lines = code_text.split('\n')
    signature_lines = []
    capturing = False
    paren_count = 0
    
    for line in lines:
        if not capturing and 'def ' in line:
            capturing = True
        
        if capturing:
            signature_lines.append(line.strip())
            # Count opening and closing parentheses
            paren_count += line.count('(') - line.count(')')
            
            # Check if we've closed all parentheses
            if paren_count == 0 and '(' in ' '.join(signature_lines):
                # Signature complete
                break
    
    # Join the lines and clean up extra spaces
    full_signature = ' '.join(signature_lines)
    
    # Remove everything after the first colon that appears after closing )
    if ')' in full_signature:
        close_paren_idx = full_signature.rfind(')')
        # Find the colon after the closing paren
        remaining = full_signature[close_paren_idx:]
        if ':' in remaining:
            colon_idx = close_paren_idx + remaining.index(':')
            full_signature = full_signature[:colon_idx].strip()
    
    return full_signature

# Extract and print full function signatures
print("Full function signatures from content_context:\n")
for i, resp in enumerate(content_context):
    full_sig = extract_full_function_signature(resp[1])
    print(f"Run {i+1}: {full_sig}")
    print()

Full function signatures from content_context:

Run 1: def linear_regression_ols( X: NDArray[np.floating], y: NDArray[np.floating], *, add_intercept: bool = True, ) -> NDArray[np.floating]

Run 2: def multiple_linear_regression( X: np.ndarray, y: np.ndarray, add_intercept: bool = True, ) -> np.ndarray

Run 3: def linear_regression_ols( X: NDArray[np.floating], y: NDArray[np.floating], *, add_intercept: bool = True, ridge_lambda: float = 0.0, return_fitted: bool = False, ) -> NDArray[np.floating] | Tuple[NDArray[np.floating], NDArray[np.floating]]

Run 4: def multiple_linear_regression( X: NDArray[np.floating], y: NDArray[np.floating], *, fit_intercept: bool = True ) -> Tuple[NDArray[np.floating], NDArray[np.floating]]

Run 5: def linear_regression_ols( X: NDArray[np.floating], y: NDArray[np.floating], *, add_intercept: bool = True, ) -> Tuple[NDArray[np.floating], NDArray[np.floating]]

Run 6: def linear_regression_ols( X: ArrayLike, y: ArrayLike, *, add_intercept: bool = True ) -> NDA

In [7]:
# Debug: let's see what the lines look like
print("Lines containing function definition from Run 1:")
lines = content_context[0][1].split('\n')
for i, line in enumerate(lines):
    if 'def ' in line or i > 0 and any('def ' in lines[j] for j in range(max(0, i-5), i)):
        print(f"Line {i}: '{line}'")

Lines containing function definition from Run 1:
Line 8: 'def linear_regression_ols('
Line 9: '    X: NDArray[np.floating],'
Line 10: '    y: NDArray[np.floating],'
Line 11: '    *,'
Line 12: '    add_intercept: bool = True,'
Line 13: ') -> NDArray[np.floating]:'


In [8]:
print("Full function signatures from content_plain:\n")
for i, resp in enumerate(content_plain):
    full_sig = extract_full_function_signature(resp[1])
    print(f"Run {i+1}: {full_sig}")
    print()


Full function signatures from content_plain:

Run 1: def multiple_linear_regression(X, y, *, add_intercept=True, method="solve")

Run 2: def multiple_linear_regression(X, y, fit_intercept=True, method="auto", rcond=None)

Run 3: def multiple_linear_regression(X, y, add_intercept=True, method="solve")

Run 4: def multiple_linear_regression(X, y, fit_intercept=True, rcond=None)

Run 5: def multiple_linear_regression(X, y, fit_intercept=True, method="solve")

Run 6: def multiple_linear_regression(X, y, add_intercept=True, method="svd", rcond=None)

Run 7: def multiple_linear_regression(X, y, add_intercept=True, method="solve")

Run 8: def multiple_linear_regression(X, y, add_intercept=True, method="auto")

Run 9: def multiple_linear_regression(X, y, add_intercept=True, method="qr", rcond=None)

Run 10: def multiple_linear_regression(X, y, add_intercept=True, method="lstsq")

Run 11: def multiple_linear_regression(X, y, add_intercept=True, method="solve")

Run 12: def multiple_linear_regre